# MotorNet-JAX: Muscle Demo

This notebook demonstrates the different muscle types available in MotorNet-JAX and how they behave.
We'll use a simple point-mass system to visualize muscle dynamics.

In [ ]:
import os
import sys

# Add parent directory to path for imports
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.')))))

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

# Import MotorNet-JAX
import motornet_jax as mn
from motornet_jax.types import JointState, MuscleState, GeometryState
from motornet_jax.skeleton import PointMass, TwoDofArm
from motornet_jax.muscle import RigidTendonMuscle, ReluMuscle

print('All packages imported.')
print(f'JAX version: {jax.__version__}')
print(f'MotorNet-JAX version: {mn.__version__}')

## I. Introduction

In this notebook, we'll explore the different muscle types in MotorNet-JAX:

1. **ReluMuscle**: Simple rectified linear muscle - force is proportional to activation
2. **RigidTendonMuscle**: Hill-type muscle with rigid tendon - includes force-length and force-velocity relationships

We'll demonstrate these using the Arm26 model, which has 6 muscles:
- Pectoralis (shoulder flexor)
- Deltoid (shoulder extensor)  
- Brachioradialis (elbow flexor)
- Triceps lateral (elbow extensor)
- Biceps (biarticular flexor)
- Triceps long (biarticular extensor)

## II. ReLU Muscles

ReLU muscles are the simplest muscle type. The force output is simply:

$$F = a \cdot F_{max}$$

where $a$ is activation (0-1) and $F_{max}$ is the maximum isometric force.

In [ ]:
# Create a ReluMuscle
relu_muscle = ReluMuscle(
    max_isometric_force=jnp.array([100.0, 200.0, 150.0]),  # 3 muscles
)
relu_params = relu_muscle.get_params()

print("ReluMuscle Parameters:")
print(f"  Max isometric force: {relu_params.max_isometric_force}")

# Test force computation at different activations
activations = jnp.array([[0.0, 0.0, 0.0],
                         [0.5, 0.5, 0.5],
                         [1.0, 1.0, 1.0]])

muscle_state = MuscleState(
    activation=activations,
    fiber_length=jnp.ones((3, 3)) * 0.1,
    fiber_velocity=jnp.zeros((3, 3)),
)

forces = ReluMuscle.compute_force(muscle_state, relu_params)

print("\nForce at different activations:")
for i, a in enumerate([0.0, 0.5, 1.0]):
    print(f"  Activation = {a}: Force = {forces[i]}")

## III. Hill-Type Muscles (Rigid Tendon)

Hill-type muscles are more realistic. They include:

1. **Force-Length Relationship**: Muscles produce maximum force at optimal length
2. **Force-Velocity Relationship**: Muscles produce less force when shortening quickly
3. **Passive Forces**: Stretched muscles produce passive elastic forces

The total force is:
$$F = (a \cdot f_l \cdot f_v + f_{passive}) \cdot F_{max}$$

In [ ]:
# Create a RigidTendonMuscle
hill_muscle = RigidTendonMuscle(
    max_isometric_force=jnp.array([100.0]),
    optimal_fiber_length=jnp.array([0.1]),
    tendon_slack_length=jnp.array([0.05]),
)
hill_params = hill_muscle.get_params()

print("RigidTendonMuscle Parameters:")
print(f"  Max isometric force: {hill_params.max_isometric_force}")
print(f"  Optimal fiber length: {hill_params.optimal_fiber_length}")
print(f"  Tendon slack length: {hill_params.tendon_slack_length}")

In [ ]:
# Plot Force-Length relationship
n_points = 100
fiber_lengths = jnp.linspace(0.05, 0.2, n_points).reshape(-1, 1)

# Create states for each length
muscle_states = MuscleState(
    activation=jnp.ones((n_points, 1)),  # Full activation
    fiber_length=fiber_lengths,
    fiber_velocity=jnp.zeros((n_points, 1)),
)

geometry_states = GeometryState(
    musculotendon_length=fiber_lengths + hill_params.tendon_slack_length,
    musculotendon_velocity=jnp.zeros((n_points, 1)),
    moment_arm=jnp.zeros((n_points, 1, 2)),
)

forces, flpe, flce, fvce = RigidTendonMuscle.compute_force(muscle_states, geometry_states, hill_params)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Normalize fiber length
normalized_length = fiber_lengths / hill_params.optimal_fiber_length

axes[0].plot(normalized_length, flce)
axes[0].set_xlabel('Normalized Fiber Length')
axes[0].set_ylabel('Active Force-Length (normalized)')
axes[0].set_title('Force-Length (Active)')
axes[0].axvline(1.0, color='r', linestyle='--', label='Optimal length')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(normalized_length, flpe)
axes[1].set_xlabel('Normalized Fiber Length')
axes[1].set_ylabel('Passive Force-Length (normalized)')
axes[1].set_title('Force-Length (Passive)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(normalized_length, forces / hill_params.max_isometric_force)
axes[2].set_xlabel('Normalized Fiber Length')
axes[2].set_ylabel('Total Force (normalized)')
axes[2].set_title('Total Force Output')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot Force-Velocity relationship
velocities = jnp.linspace(-1.0, 0.5, n_points).reshape(-1, 1)

muscle_states = MuscleState(
    activation=jnp.ones((n_points, 1)),
    fiber_length=jnp.ones((n_points, 1)) * hill_params.optimal_fiber_length,
    fiber_velocity=velocities,
)

geometry_states = GeometryState(
    musculotendon_length=jnp.ones((n_points, 1)) * (hill_params.optimal_fiber_length + hill_params.tendon_slack_length),
    musculotendon_velocity=velocities,
    moment_arm=jnp.zeros((n_points, 1, 2)),
)

forces, _, _, fvce = RigidTendonMuscle.compute_force(muscle_states, geometry_states, hill_params)

# Normalize velocity by max velocity (typically 10 * optimal_length)
max_velocity = 10.0 * hill_params.optimal_fiber_length
normalized_velocity = velocities / max_velocity

plt.figure(figsize=(8, 5))
plt.plot(normalized_velocity, fvce)
plt.xlabel('Normalized Velocity (shortening = negative)')
plt.ylabel('Force-Velocity Factor')
plt.title('Force-Velocity Relationship')
plt.axhline(1.0, color='r', linestyle='--', alpha=0.5, label='Isometric')
plt.axvline(0.0, color='gray', linestyle='--', alpha=0.5)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## IV. Arm26 Model Demonstration

Now let's use the full Arm26 model to simulate movements with different muscle activation patterns.

In [ ]:
from motornet_jax.effector import Arm26

# Create Arm26 effector
arm = Arm26(dt=0.01, n_ministeps=1)
params = arm.get_params()

print("Arm26 Configuration:")
print(f"  Number of muscles: {arm.n_muscles}")
print(f"  Muscle names: {arm.muscle_names}")
print(f"  Timestep: {params.dt}s")
print(f"\nMax isometric forces (N):")
for name, force in zip(arm.muscle_names, params.max_isometric_force):
    print(f"  {name}: {force:.0f}")

In [ ]:
# Define simulation function
@partial(jax.jit, static_argnums=(4,))
def simulate(state, action, endpoint_load, joint_load, n_steps):
    """Simulate arm movement and record trajectory."""
    def step_fn(state, _):
        new_state = Arm26.step(state, action, endpoint_load, joint_load, params)
        return new_state, {
            'fingertip': new_state.fingertip[0],
            'joint_pos': new_state.joint.position[0],
            'activation': new_state.muscle.activation[0],
            'muscle_force': Arm26.compute_muscle_force(new_state.muscle.activation, new_state.geometry, params)[0],
        }
    
    final_state, trajectory = jax.lax.scan(step_fn, state, None, length=n_steps)
    return final_state, trajectory

# Initial joint state
initial_joint = JointState(
    position=jnp.array([[0.8, 0.9]]),  # Shoulder and elbow angles
    velocity=jnp.zeros((1, 2)),
)

n_steps = 200
endpoint_load = jnp.zeros((1, 2))
joint_load = jnp.zeros((1, 2))

In [ ]:
# Define different activation patterns
activation_patterns = {
    'Shoulder Flexor (Pectoralis)': jnp.array([[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]]),
    'Shoulder Extensor (Deltoid)': jnp.array([[0.0, 1.0, 0.0, 0.0, 0.0, 0.0]]),
    'Elbow Flexor (Brachioradialis)': jnp.array([[0.0, 0.0, 1.0, 0.0, 0.0, 0.0]]),
    'Elbow Extensor (Triceps Lat)': jnp.array([[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]]),
    'All Flexors': jnp.array([[1.0, 0.0, 1.0, 0.0, 1.0, 0.0]]),
    'All Extensors': jnp.array([[0.0, 1.0, 0.0, 1.0, 0.0, 1.0]]),
    'Co-contraction': jnp.array([[0.5, 0.5, 0.5, 0.5, 0.5, 0.5]]),
}

# Simulate each pattern
results = {}
for name, action in activation_patterns.items():
    state = arm.reset(batch_size=1, joint_state=initial_joint)
    final_state, trajectory = simulate(state, action, endpoint_load, joint_load, n_steps)
    results[name] = trajectory
    
print("Simulations complete!")

In [ ]:
# Plot hand trajectories
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Hand trajectory in Cartesian space
ax = axes[0, 0]
for name, traj in results.items():
    ax.plot(traj['fingertip'][:, 0], traj['fingertip'][:, 1], label=name, linewidth=2)
ax.scatter([results['Shoulder Flexor (Pectoralis)']['fingertip'][0, 0]], 
           [results['Shoulder Flexor (Pectoralis)']['fingertip'][0, 1]], 
           c='black', s=100, marker='o', zorder=5, label='Start')
ax.set_xlabel('X position (m)')
ax.set_ylabel('Y position (m)')
ax.set_title('Hand Trajectories')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# Shoulder angle over time
ax = axes[0, 1]
t = np.arange(n_steps) * params.dt
for name, traj in results.items():
    ax.plot(t, np.rad2deg(traj['joint_pos'][:, 0]), label=name, linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Shoulder angle (deg)')
ax.set_title('Shoulder Joint Angle')
ax.grid(True, alpha=0.3)

# Elbow angle over time
ax = axes[1, 0]
for name, traj in results.items():
    ax.plot(t, np.rad2deg(traj['joint_pos'][:, 1]), label=name, linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Elbow angle (deg)')
ax.set_title('Elbow Joint Angle')
ax.grid(True, alpha=0.3)

# Muscle forces for one pattern
ax = axes[1, 1]
pattern = 'All Flexors'
for i, muscle_name in enumerate(arm.muscle_names):
    ax.plot(t, results[pattern]['muscle_force'][:, i], label=muscle_name, linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Muscle Force (N)')
ax.set_title(f'Muscle Forces ({pattern})')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## V. Activation Dynamics

Muscle activation doesn't change instantaneously - it follows first-order dynamics with different time constants for activation and deactivation.

In [ ]:
# Demonstrate activation dynamics
n_steps = 300

# Step input: excitation goes from 0 to 1 at t=50, back to 0 at t=150
def simulate_activation_dynamics(state, params, n_steps):
    activations = []
    
    for step in range(n_steps):
        # Excitation signal
        if 50 <= step < 150:
            action = jnp.ones((1, 6))
        else:
            action = jnp.zeros((1, 6))
        
        state = Arm26.step(state, action, jnp.zeros((1, 2)), jnp.zeros((1, 2)), params)
        activations.append(state.muscle.activation[0, 0])
    
    return jnp.array(activations)

state = arm.reset(batch_size=1, joint_state=initial_joint)
activations = simulate_activation_dynamics(state, params, n_steps)

t = np.arange(n_steps) * params.dt
excitation = np.zeros(n_steps)
excitation[50:150] = 1.0

plt.figure(figsize=(10, 5))
plt.plot(t, excitation, 'k--', linewidth=2, label='Excitation (input)')
plt.plot(t, activations, 'b-', linewidth=2, label='Activation (output)')
plt.xlabel('Time (s)')
plt.ylabel('Value')
plt.title('Muscle Activation Dynamics')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-0.1, 1.1)

# Annotate time constants
plt.annotate(f'τ_act = {params.tau_activation*1000:.0f} ms', 
             xy=(0.6, 0.5), fontsize=12)
plt.annotate(f'τ_deact = {params.tau_deactivation*1000:.0f} ms', 
             xy=(1.8, 0.5), fontsize=12)

plt.show()

## VI. Performance Comparison

One of the key advantages of MotorNet-JAX is speed. Let's benchmark the simulation throughput.

In [ ]:
import time

# Benchmark
batch_size = 64
n_steps = 100
n_episodes = 1000

key = jax.random.PRNGKey(0)
state = arm.reset(batch_size=batch_size, key=key)
action = jnp.ones((batch_size, 6)) * 0.3
endpoint_load = jnp.zeros((batch_size, 2))
joint_load = jnp.zeros((batch_size, 2))

@jax.jit
def run_episode(state):
    def step_fn(state, _):
        new_state = Arm26.step(state, action, endpoint_load, joint_load, params)
        return new_state, None
    final_state, _ = jax.lax.scan(step_fn, state, None, length=n_steps)
    return final_state

# Warmup
state = arm.reset(batch_size=batch_size, key=key)
_ = run_episode(state)
_.joint.position.block_until_ready()

# Benchmark
start = time.perf_counter()
for _ in range(n_episodes):
    state = arm.reset(batch_size=batch_size, key=key)
    final = run_episode(state)
final.joint.position.block_until_ready()
elapsed = time.perf_counter() - start

total_steps = n_episodes * n_steps * batch_size
steps_per_sec = total_steps / elapsed
episodes_per_sec = n_episodes / elapsed

print(f"\nPerformance Results:")
print(f"  Batch size: {batch_size}")
print(f"  Steps per episode: {n_steps}")
print(f"  Total episodes: {n_episodes}")
print(f"  Time elapsed: {elapsed:.2f}s")
print(f"\n  Steps per second: {steps_per_sec:,.0f}")
print(f"  Episodes per second: {episodes_per_sec:,.1f}")

## Summary

In this notebook we covered:

1. **ReluMuscle**: Simple linear force-activation relationship
2. **RigidTendonMuscle**: Hill-type muscle with:
   - Force-length relationship (peak at optimal length)
   - Force-velocity relationship (reduced force during shortening)
   - Passive elastic forces
3. **Arm26 Model**: 6-muscle arm with realistic muscle dynamics
4. **Activation Dynamics**: First-order dynamics with different time constants
5. **Performance**: JAX enables millions of simulation steps per second

Key differences from PyTorch MotorNet:
- All functions are pure and JIT-compiled
- State is passed explicitly (not stored in objects)
- Use `jax.lax.scan` for efficient trajectory rollouts
- Automatic vectorization with `jax.vmap`